In [9]:
import pandas as pd
import numpy as np

In [20]:
import matplotlib.pyplot as plt

In [91]:
name = 'train_ALL+reliability'
train = pd.read_csv(f'data/train_ALL+reliability.csv')
test = pd.read_csv(f'data/test_ALL+reliability.csv')

In [92]:
test.columns

Index(['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity',
       'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'pet',
       '_merge_terra', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI',
       '_merge_landsat', 'Impute_Method', 'geometry', 'STAT_ID', 'sc', 'ss',
       'su', 'mt', 'va', 'vb', 'vi', 'pa', 'pb', 'pi', 'GLC_Artificial',
       'GLC_Managed', 'GLC_Water', 'GLC_Aquatic_Veg', 'GLC_PERC_COV',
       'Popdens_00', 'Soil_pH', 'SOC', 'Soil_wetness', 'dist_m', 'dist_km',
       'Latitude_glorich', 'Longitude_glorich', 'date', 'Alkalinity', 'Cl',
       'DIP', 'SO4', 'SpecCond25C', 'pH', 'Alkalinity_reliability',
       'Cl_reliability', 'DIP_reliability', 'SO4_reliability',
       'SpecCond25C_reliability', 'pH_reliability', 'date_diff_days',
       'dws_1st', 'dws_1st_dist', 'dws_TAL', 'dws_EC', 'dws_pH', 'dws_Ca',
       'dws_Mg', 'dws_Na', 'dws_Cl', 'dws_SO4', 'dws_days_diff', 'dws_dist_km',
       'dws_P_modified', 'dws_P_reliability', 'month', '

In [97]:
train.head()
df= test.copy()

In [98]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 77 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Latitude                       200 non-null    float64
 1   Longitude                      200 non-null    float64
 2   Sample Date                    200 non-null    object 
 3   Total Alkalinity               0 non-null      float64
 4   Electrical Conductance         0 non-null      float64
 5   Dissolved Reactive Phosphorus  0 non-null      float64
 6   pet                            200 non-null    float64
 7   _merge_terra                   200 non-null    object 
 8   nir                            200 non-null    float64
 9   green                          200 non-null    float64
 10  swir16                         200 non-null    float64
 11  swir22                         200 non-null    float64
 12  NDMI                           200 non-null    flo

In [99]:
df['Sample Date'] = pd.to_datetime(df['Sample Date'],format = 'mixed')
df['month'] = df['Sample Date'].dt.month
df['day_of_year'] = df['Sample Date'].dt.dayofyear

# === Cyclic encoding (best for tree + linear models) ===
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

# === South Africa wet/dry season flag ===
df['wet_season'] = df['month'].isin([10, 11, 12, 1, 2, 3]).astype(int)

In [100]:
df.to_csv('data/test_ALL+reliability.csv', index=False)

# data analysis 

In [103]:
# everything whack brute force
s57 = pd.read_csv('data/best_submission.csv')
s57.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,2014-09-01,102.082,287.023260,20.126828
1,-33.329167,26.077500,2015-09-16,87.910,750.317075,19.268930
2,-32.991639,27.640028,2015-05-07,69.618,380.191300,19.731278
3,-34.096389,24.439167,2012-02-07,42.467,512.783735,13.437366
4,-32.000556,28.581667,2014-10-01,84.051,234.930341,20.088077


In [104]:
s57.describe()

,Latitude,Longitude,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
count,200.000000,200.000000,200.000000,200.000000,200.000000
mean,-32.816731,26.583118,128.076290,496.580275,29.562633
std,0.652012,1.322303,107.837373,285.648369,20.774526
min,-34.096389,24.196389,17.862000,103.597023,9.719955
25%,-33.185361,25.430000,65.837750,292.035534,19.517124
50%,-32.991639,27.366667,94.841000,399.254560,20.289043
75%,-32.086390,27.640028,136.710750,658.148042,36.513065
max,-31.903056,28.581667,514.185000,1335.998076,105.016921


In [105]:
# model others, copy paste alkalinity 
s55 = pd.read_csv('data/s55.csv')
s55.describe()

,Latitude,Longitude,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
count,200.000000,200.000000,200.000000,200.000000,200.000000
mean,-32.816731,26.583118,128.076290,485.568776,28.989169
std,0.652012,1.322303,107.837373,268.897155,22.773854
min,-34.096389,24.196389,17.862000,105.131989,8.335373
25%,-33.185361,25.430000,65.837750,292.735740,13.996935
50%,-32.991639,27.366667,94.841000,400.098209,18.255080
75%,-32.086390,27.640028,136.710750,649.651295,37.489838
max,-31.903056,28.581667,514.185000,1209.728090,116.604782


In [107]:
s13 = pd.read_csv('data/s13.csv')
s13.describe()

,Latitude,Longitude,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
count,200.000000,200.000000,200.000000,200.000000,200.000000
mean,-32.816731,26.583118,128.076290,485.568776,128.076290
std,0.652012,1.322303,107.837373,268.897155,107.837373
min,-34.096389,24.196389,17.862000,105.131989,17.862000
25%,-33.185361,25.430000,65.837750,292.735740,65.837750
50%,-32.991639,27.366667,94.841000,400.098209,94.841000
75%,-32.086390,27.640028,136.710750,649.651295,136.710750
max,-31.903056,28.581667,514.185000,1209.728090,514.185000


In [108]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9319 entries, 0 to 9318
Data columns (total 78 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Latitude                       9319 non-null   float64
 1   Longitude                      9319 non-null   float64
 2   Sample Date                    9319 non-null   object 
 3   Total Alkalinity               9319 non-null   float64
 4   Electrical Conductance         9319 non-null   float64
 5   Dissolved Reactive Phosphorus  9319 non-null   float64
 6   pet                            9319 non-null   float64
 7   _merge_terra                   9319 non-null   object 
 8   nir                            9319 non-null   float64
 9   green                          9319 non-null   float64
 10  swir16                         9319 non-null   float64
 11  swir22                         9319 non-null   float64
 12  NDMI                           9319 non-null   f